# Distributed BFS (Simulation Notes)

Goal: capture the core ideas behind *distributed* breadth-first search (BFS) on a graph, and provide a small runnable simulation of message-passing BFS.

We will model a simple Bulk Synchronous Parallel (BSP) style algorithm:

- Graph is partitioned across workers.
- Each superstep, workers process their local frontier and send discovery messages to other workers.
- A node is first assigned the smallest level (distance) seen.


## Why "distributed" BFS is different

Sequential BFS is straightforward: maintain a queue (or frontier) and a visited set.

In distributed BFS:

- The graph is split: each worker owns a subset of vertices (and their adjacency lists).
- "Visited" is distributed state: ownership matters for correctness.
- Edge traversals can become network messages.
- Work happens in waves (levels), which maps nicely to BSP supersteps.

Performance drivers:

- Communication volume: cross-partition edges dominate.
- Synchronization: each level typically implies a barrier.
- Load balance: frontier sizes can be skewed.


In [ ]:
from __future__ import annotations

from collections import defaultdict, deque
from dataclasses import dataclass
from typing import Dict, Iterable, List, Set, Tuple


Graph = Dict[str, List[str]]


def make_example_graph() -> Graph:
    # Undirected-ish (we store both directions for simplicity).
    edges = [
        ("A", "B"),
        ("A", "C"),
        ("B", "D"),
        ("C", "D"),
        ("C", "E"),
        ("D", "F"),
        ("E", "F"),
        ("E", "G"),
        ("F", "H"),
    ]

    g: Graph = defaultdict(list)
    for u, v in edges:
        g[u].append(v)
        g[v].append(u)
    return dict(g)


G = make_example_graph()
G

In [ ]:
def bfs_sequential(g: Graph, source: str) -> Dict[str, int]:
    dist: Dict[str, int] = {source: 0}
    q: deque[str] = deque([source])
    while q:
        u = q.popleft()
        for v in g.get(u, []):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist


bfs_sequential(G, "A")

## A minimal distributed/BSP BFS simulation

We simulate ownership by assigning each vertex to a worker. Messages are `(vertex, proposed_distance)` sent to the owning worker.

Algorithm (per superstep):

1. Deliver messages to workers.
2. Workers accept the first (smallest) distance for each newly discovered local vertex.
3. Newly discovered vertices expand: for each neighbor, send a message with `dist + 1`.
4. Repeat until no messages are in flight.


In [ ]:
@dataclass(frozen=True)
class Msg:
    v: str
    dist: int


def owner(partition: Dict[str, int], v: str) -> int:
    return partition[v]


def round_robin_partition(vertices: Iterable[str], k: int) -> Dict[str, int]:
    vs = sorted(vertices)
    return {v: i % k for i, v in enumerate(vs)}


def bfs_distributed_bsp(
    g: Graph,
    source: str,
    k_workers: int,
    partition: Dict[str, int] | None = None,
    max_steps: int = 10_000,
) -> Tuple[Dict[str, int], Dict[int, int]]:
    if partition is None:
        partition = round_robin_partition(g.keys(), k_workers)

    # Per-worker distance map for owned vertices.
    dist: Dict[str, int] = {}

    # Messages in flight, grouped by destination worker.
    inbox: Dict[int, List[Msg]] = {w: [] for w in range(k_workers)}

    # Seed with the source.
    inbox[owner(partition, source)].append(Msg(source, 0))

    steps = 0
    stats_msgs_by_step: Dict[int, int] = {}

    while steps < max_steps:
        steps += 1

        total_msgs = sum(len(m) for m in inbox.values())
        stats_msgs_by_step[steps] = total_msgs
        if total_msgs == 0:
            break

        next_inbox: Dict[int, List[Msg]] = {w: [] for w in range(k_workers)}

        # Superstep: deliver + process.
        for w in range(k_workers):
            msgs = inbox[w]
            if not msgs:
                continue

            # Accept best (smallest) proposal per vertex this step.
            best: Dict[str, int] = {}
            for m in msgs:
                cur = best.get(m.v)
                if cur is None or m.dist < cur:
                    best[m.v] = m.dist

            # Newly discovered local vertices expand.
            for v, d in best.items():
                if v in dist:
                    # Already finalized earlier.
                    continue
                dist[v] = d
                for nbr in g.get(v, []):
                    dest = owner(partition, nbr)
                    next_inbox[dest].append(Msg(nbr, d + 1))

        inbox = next_inbox

    return dist, stats_msgs_by_step


partition = round_robin_partition(G.keys(), k=3)
partition

In [ ]:
dist_d, msg_stats = bfs_distributed_bsp(G, source="A", k_workers=3, partition=partition)
dist_d, msg_stats

In [ ]:
# Sanity check: distances should match sequential BFS on an unweighted graph.
dist_seq = bfs_sequential(G, "A")
assert dist_d == dist_seq
dist_seq

## Notes and extensions

- Partition quality matters: fewer cross edges usually means fewer network messages.
- Many real systems add *message combiner* logic to reduce duplicates (we did a tiny per-step combiner with `best`).
- For huge graphs, storing adjacency lists and frontiers efficiently dominates runtime.
- Async variants exist (no per-level barrier), but need careful reasoning about duplicates and termination detection.

Exercises:

1. Change `round_robin_partition` to a hand-crafted partition and compare `msg_stats`.
2. Add a counter for "cross-worker" messages only.
3. Implement an asynchronous simulation (workers pop from inboxes until empty) and compare step behavior.
